# 04 - Feature Engineering

## Objective

Feature engineering creates new variables from the existing dataset to improve the machine learning model's ability to identify fraudulent transactions.

The engineered features must only use information that is available at the time a transaction is processed.

In this notebook we will:

- Create amount categories
- Create age groups
- Create transaction time categories
- Create high velocity indicator
- Create low device trust indicator
- Compare engineered features with original features
- Save the engineered dataset

In [ ]:
import pandas as pd
import numpy as np

print("Libraries Imported Successfully")

In [ ]:
df = pd.read_csv("../data/processed/processed_transactions.csv")

df.head()

In [ ]:
print("Dataset Shape:", df.shape)

df.info()

In [ ]:
df["amount_category"] = pd.cut(

    df["amount"],

    bins=[0,100,500,1000,np.inf],

    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ]

)

df[["amount","amount_category"]].head()

In [ ]:
df["age_group"] = pd.cut(

    df["cardholder_age"],

    bins=[18,30,45,60,100],

    labels=[
        "Young",
        "Adult",
        "Middle Age",
        "Senior"
    ]

)

df[["cardholder_age","age_group"]].head()

In [ ]:
def transaction_period(hour):

    if hour < 6:
        return "Night"

    elif hour < 12:
        return "Morning"

    elif hour < 18:
        return "Afternoon"

    else:
        return "Evening"

df["transaction_period"] = df["transaction_hour"].apply(transaction_period)

df[["transaction_hour","transaction_period"]].head()

In [ ]:
median_velocity = df["velocity_last_24h"].median()

df["high_velocity"] = np.where(

    df["velocity_last_24h"] > median_velocity,

    1,

    0

)

df[["velocity_last_24h","high_velocity"]].head()

In [ ]:
median_trust = df["device_trust_score"].median()

df["low_device_trust"] = np.where(

    df["device_trust_score"] < median_trust,

    1,

    0

)

df[["device_trust_score","low_device_trust"]].head()

In [ ]:
df["foreign_location_risk"] = (

    df["foreign_transaction"]

    *

    df["location_mismatch"]

)

df[

    [

        "foreign_transaction",

        "location_mismatch",

        "foreign_location_risk"

    ]

].head()

In [ ]:
df["log_amount"] = np.log1p(df["amount"])

df[["amount","log_amount"]].head()

In [ ]:
new_features = [

    "amount_category",

    "age_group",

    "transaction_period",

    "high_velocity",

    "low_device_trust",

    "foreign_location_risk",

    "log_amount"

]

df[new_features].head()

In [ ]:
print(df.shape)

df.info()

In [ ]:
df.to_csv(

    "../data/processed/engineered_transactions.csv",

    index=False

)

print("Engineered Dataset Saved Successfully")

# Engineered Features

The following new features were created:

## 1. amount_category

Groups transaction amounts into:

- Low
- Medium
- High
- Very High

---

## 2. age_group

Groups cardholders by age.

---

## 3. transaction_period

Categorizes transactions into:

- Night
- Morning
- Afternoon
- Evening

---

## 4. high_velocity

Indicates whether the number of transactions within the last 24 hours is above the median.

---

## 5. low_device_trust

Identifies transactions performed on devices with below-median trust scores.

---

## 6. foreign_location_risk

Represents transactions that are both foreign and have a location mismatch.

---

## 7. log_amount

Reduces skewness in transaction amounts using logarithmic transformation.

# Comparison with Original Dataset

Original Features

- amount
- transaction_hour
- merchant_category
- foreign_transaction
- location_mismatch
- device_trust_score
- velocity_last_24h
- cardholder_age

New Features

- amount_category
- age_group
- transaction_period
- high_velocity
- low_device_trust
- foreign_location_risk
- log_amount

These engineered features provide additional information that may improve fraud detection performance while using only information available at prediction time.